# Apple Depth Pro 2 정리: Depth Pro와 달라진 점(영상·프레임 간 일관성), S~H+ 모델 크기, 연구용 라이선스, 사진·영상 깊이 지도 뽑아 보기

위에서부터 차례로 실행해 보세요.

### 저장소 받고 CPU용 torch와 함께 설치

In [ ]:
%%bash
set -eo pipefail
cd /content
git clone https://github.com/apple-aiml-research/ml-depth-pro-2
python3 -m venv /content/venv
. /content/venv/bin/activate
pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cpu
pip install -q /content/ml-depth-pro-2
pip freeze | egrep -i '^(torch|torchvision|timm|av)=='

### S 모델로 사진 한 장 깊이 추정(가중치 자동 다운로드)

In [ ]:
%%bash
set -eo pipefail
. /content/venv/bin/activate
cd /content/ml-depth-pro-2
depth-pro-2 predict -i assets/example.jpg -o /content/out_photo -m depthpro2-s --save-npz
python3 -I - <<'E'
import numpy as np
d=np.load('/content/out_photo/example-depth.npz'); x=d['depth']; f=np.isfinite(x)
print(x.shape, d['focallength_px'], 'finite min', x[f].min(), 'median', np.median(x[f]), 'inf frac', 1-f.mean())
E

### 영상을 480x360으로 줄이기(메모리 절약)

In [ ]:
%%bash
set -eo pipefail
mkdir -p /content/inputs_lo
ffmpeg -v error -y -i /content/ml-depth-pro-2/assets/example.mp4 -vf scale=480:360 -an -c:v libx264 -pix_fmt yuv420p /content/inputs_lo/example_480.mp4
ls -l /content/inputs_lo

### buffered 영상 모드로 깊이 영상 만들기

In [ ]:
%%bash
set -eo pipefail
. /content/venv/bin/activate
time depth-pro-2 predict -i /content/inputs_lo/example_480.mp4 -o /content/out_video -m depthpro2-s --save-npz

### 프레임별 추론 스크립트 만들기

In [ ]:
%%bash
cat > /content/perframe.py <<'E'
import av, time, numpy as np, torch
from depth_pro_2 import create_model_and_transforms, upright
model, transform = create_model_and_transforms("depthpro2-s")
video = av.open("/content/inputs_lo/example_480.mp4")
out=[]; t=time.time()
for i,f in enumerate(video.decode(video=0)):
    d = model.infer(transform(upright(f)), f_px=393.6)["depth"].float().cpu().numpy()
    out.append(d)
print("total", time.time()-t)
np.save("/content/perframe_depth.npy", np.stack(out).astype(np.float32))
E

### 프레임별 vs buffered 간이 깜빡임 지표 비교

In [ ]:
%%bash
set -eo pipefail
. /content/venv/bin/activate
python3 -I /content/perframe.py
python3 -I - <<'E'
import numpy as np, glob
fs=sorted(glob.glob('/content/out_video/example_480-depth/*.npz'))
B=np.stack([np.load(f)['depth'] for f in fs]); P=np.load('/content/perframe_depth.npy')
n=min(len(B),len(P)); B=B[:n]; P=P[:n]
j=lambda D: np.abs(np.diff(np.log(np.clip(D,0.5,80)),axis=0)).mean()
print('buffered', j(B), 'per-frame', j(P))
E